# Versioned SER Result Summary

Notebook gọn để tổng hợp tất cả `cross_session_summary.json` thành bảng mean ± std và heatmap theo held-out session. Notebook này không training, chỉ đọc kết quả đã có.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib"))

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    display
except NameError:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid", context="paper")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"notebook", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / "results"
OUT_DIR = PROJECT_ROOT / "reports" / "versioned_result_summary"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["WA", "UA", "WF1", "Macro-F1"]
PRIMARY_METRIC = "Macro-F1"
print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUT_DIR      =", OUT_DIR)


## 2. Load Cross-Session Summaries


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def pct(value: float) -> float:
    return 100.0 * float(value)


def mean_std_text(mean: float, std: float) -> str:
    return f"{pct(mean):.2f} ± {pct(std):.2f}"


def infer_family(path: Path) -> str:
    parts = path.parts
    for key in ["versioned_loso", "ablation_model_loso", "ablation_loso", "tim_ablations"]:
        if key in parts:
            return key
    if "cross_session" in parts:
        return "legacy_loso"
    return "other"


def infer_model_name(path: Path, payload: dict) -> str:
    text = str(path)
    trainer = str(payload.get("trainer_module", ""))
    if "tim_zero" in text:
        return "TIM-zero"
    if "tim_shuffled" in text:
        return "TIM-shuffled"
    if "tim_no_" in text or "tim_overlap_only" in text:
        return path.parents[2].name if "tim_ablations" in path.parts else path.parents[3].name
    if "v3_2_tim_compact_primitives" in text:
        return "TIM v3.2 compact primitives"
    if "v3_1_tim_recommended_v2" in text or "wavlm_tim_v2_recommended" in text:
        return "TIM v3.1 recommended-v2"
    if "v2_2_2_dual_temporal_dialogue_fuse" in text or "wavlm_dual_branch_temporal_first" in text:
        return "Dual v2.2.2 temporal-first"
    if "v2_2_1_dual_dialogue_temporal_fuse" in text or "dual_branch_3phase" in text or "wavlm_dual_branch/cross_session" in text:
        return "Dual v2.2.1 3-phase"
    if "v2_1_dual_end2end" in text or "wavlm_dual_branch_end2end" in text:
        return "Dual v2.1 end-to-end"
    if "v1_tim_concat" in text or "wavlm_tim_end2end" in text or "wavlm_tim_loso" in text or trainer.endswith("train_wavlm_tim"):
        return "TIM v1 concat"
    if "wavlm_mal" in text or trainer.endswith("train_wavlm_mal"):
        return "MAL"
    if "wavlm_baseline" in text or trainer.endswith("train_wavlm_baseline"):
        return "Baseline"
    return path.parents[2].name if len(path.parents) > 2 else path.stem


def infer_version(path: Path, model_name: str) -> str:
    if "v3.1" in model_name:
        return "3.1"
    if "v2.2.2" in model_name:
        return "2.2.2"
    if "v2.2.1" in model_name:
        return "2.2.1"
    if "v2.1" in model_name:
        return "2.1"
    if "v1" in model_name:
        return "1"
    return "control"


def infer_setting(path: Path, payload: dict) -> str:
    cfg = payload.get("config", {}) if isinstance(payload.get("config"), dict) else {}
    text = str(path)
    if "setting_B" in text:
        return "B"
    if "setting_A" in text:
        return "A"
    # Existing results are frozen/precomputed fair runs unless config says otherwise.
    return "A"


def seed_from_path(path: Path) -> int | None:
    m = re.search(r"seed_(\d+)", str(path))
    return int(m.group(1)) if m else None

summary_paths = sorted(RESULTS_DIR.glob("**/cross_session_summary.json"))
aggregate_rows = []
fold_rows = []

for path in summary_paths:
    payload = read_json(path)
    family = infer_family(path)
    model_name = infer_model_name(path, payload)
    version = infer_version(path, model_name)
    setting = infer_setting(path, payload)
    seed = seed_from_path(path)
    run_name = path.parent.name
    for metric in METRICS:
        item = payload.get("aggregate", {}).get(metric)
        if not item:
            continue
        aggregate_rows.append({
            "family": family,
            "setting": setting,
            "version": version,
            "model": model_name,
            "seed": seed,
            "run_name": run_name,
            "metric": metric,
            "mean": float(item["mean"]),
            "std": float(item["std"]),
            "n": int(item.get("n", 5)),
            "summary_path": str(path),
        })
    for fold in payload.get("folds", []):
        for metric in METRICS:
            if metric not in fold.get("metrics", {}):
                continue
            fold_rows.append({
                "family": family,
                "setting": setting,
                "version": version,
                "model": model_name,
                "seed": seed,
                "run_name": run_name,
                "test_session": int(fold["test_session"]),
                "metric": metric,
                "value": float(fold["metrics"][metric]),
                "summary_path": str(path),
            })

aggregate = pd.DataFrame(aggregate_rows)
folds = pd.DataFrame(fold_rows)
aggregate.to_csv(OUT_DIR / "all_cross_session_aggregate_long.csv", index=False)
folds.to_csv(OUT_DIR / "all_cross_session_folds_long.csv", index=False)
print("summaries found:", len(summary_paths))
print("aggregate rows:", len(aggregate))
display(aggregate.head())


## 3. Main Model Table


In [ ]:
MAIN_MODEL_ORDER = [
    "Baseline",
    "MAL",
    "TIM v1 concat",
    "Dual v2.1 end-to-end",
    "Dual v2.2.1 3-phase",
    "Dual v2.2.2 temporal-first",
    "TIM v3.1 recommended-v2",
]

main = aggregate[
    aggregate["family"].isin(["ablation_model_loso", "ablation_loso", "legacy_loso", "versioned_loso"])
    & aggregate["model"].isin(MAIN_MODEL_ORDER)
].copy()
# If duplicate copies exist in ablation_loso and ablation_model_loso, prefer ablation_model_loso, then versioned_loso, then legacy.
family_priority = {"versioned_loso": 0, "ablation_model_loso": 1, "ablation_loso": 2, "legacy_loso": 3}
main["family_priority"] = main["family"].map(family_priority).fillna(9)
main = main.sort_values(["model", "metric", "family_priority", "run_name"], ascending=[True, True, True, False]).drop_duplicates(["setting", "model", "metric"], keep="first")

main_table = (
    main.assign(mean_std=lambda df: [mean_std_text(m, s) for m, s in zip(df["mean"], df["std"])])
    .pivot_table(index=["setting", "version", "model"], columns="metric", values="mean_std", aggfunc="first")
)
main_table = main_table.reindex(columns=METRICS)
ordered_index = []
for setting in sorted(main["setting"].unique()):
    for model in MAIN_MODEL_ORDER:
        matches = [idx for idx in main_table.index if idx[0] == setting and idx[2] == model]
        ordered_index.extend(matches)
main_table = main_table.loc[ordered_index]
main_table.to_csv(OUT_DIR / "main_model_summary_table.csv")
display(main_table)


## 4. Main Model Heatmaps


In [ ]:
for metric in ["Macro-F1", "UA", "WA"]:
    metric_df = main[main["metric"].eq(metric)].copy()
    if metric_df.empty:
        continue
    heat = metric_df.pivot_table(index="model", columns="setting", values="mean", aggfunc="max") * 100.0
    heat = heat.reindex([m for m in MAIN_MODEL_ORDER if m in heat.index])
    heat.to_csv(OUT_DIR / f"main_model_{metric.lower().replace('-', '_')}_by_setting.csv")
    fig, ax = plt.subplots(figsize=(7, max(3, 0.45 * len(heat))))
    sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"{metric} mean by model/setting (%)")
    ax.set_xlabel("Setting")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"main_model_{metric.lower().replace('-', '_')}_heatmap.png", dpi=180)
    plt.show()


## 5. Held-Out Session Heatmap


In [ ]:
main_folds = folds[
    folds["model"].isin(MAIN_MODEL_ORDER)
    & folds["metric"].eq(PRIMARY_METRIC)
    & folds["family"].isin(["versioned_loso", "ablation_model_loso", "ablation_loso", "legacy_loso"])
].copy()
main_folds["family_priority"] = main_folds["family"].map(family_priority).fillna(9)
main_folds = main_folds.sort_values(["model", "test_session", "family_priority", "run_name"], ascending=[True, True, True, False]).drop_duplicates(["setting", "model", "test_session"], keep="first")

for setting, frame in main_folds.groupby("setting"):
    pivot = frame.pivot_table(index="model", columns="test_session", values="value", aggfunc="mean") * 100.0
    pivot = pivot.reindex([m for m in MAIN_MODEL_ORDER if m in pivot.index])
    pivot.to_csv(OUT_DIR / f"main_model_{PRIMARY_METRIC.lower().replace('-', '_')}_by_session_setting_{setting}.csv")
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(pivot))))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"{PRIMARY_METRIC} by held-out session - Setting {setting} (%)")
    ax.set_xlabel("Held-out session")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"main_model_{PRIMARY_METRIC.lower().replace('-', '_')}_by_session_setting_{setting}.png", dpi=180)
    plt.show()

display(main_folds.head())


## 6. TIM Ablation Tables


In [ ]:
ablation = aggregate[aggregate["family"].eq("tim_ablations")].copy()
if not ablation.empty:
    ablation_table = (
        ablation.assign(mean_std=lambda df: [mean_std_text(m, s) for m, s in zip(df["mean"], df["std"])])
        .pivot_table(index="model", columns="metric", values="mean_std", aggfunc="first")
        .reindex(columns=METRICS)
    )
    ablation_table.to_csv(OUT_DIR / "tim_ablation_summary_table.csv")
    display(ablation_table)

    ablation_heat = ablation[ablation["metric"].eq(PRIMARY_METRIC)].pivot_table(index="model", columns="setting", values="mean", aggfunc="max") * 100.0
    ablation_heat.to_csv(OUT_DIR / f"tim_ablation_{PRIMARY_METRIC.lower().replace('-', '_')}_heatmap.csv")
    fig, ax = plt.subplots(figsize=(5, max(3, 0.45 * len(ablation_heat))))
    sns.heatmap(ablation_heat, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"TIM ablations {PRIMARY_METRIC} (%)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"tim_ablation_{PRIMARY_METRIC.lower().replace('-', '_')}_heatmap.png", dpi=180)
    plt.show()
else:
    print("No TIM ablation summaries found.")


## 7. Markdown Report


In [ ]:
report = f"""
# Versioned Result Summary

## Main model table

{main_table.to_markdown() if not main_table.empty else 'No main model results found.'}

## TIM ablation table

{ablation_table.to_markdown() if 'ablation_table' in globals() and not ablation_table.empty else 'No TIM ablation results found.'}

## Generated files

- `all_cross_session_aggregate_long.csv`
- `all_cross_session_folds_long.csv`
- `main_model_summary_table.csv`
- `main_model_macro_f1_by_setting.csv`
- `main_model_macro_f1_by_session_setting_A.csv`
- `tim_ablation_summary_table.csv`
- figures under `figures/`
"""
(OUT_DIR / "versioned_result_summary.md").write_text(report.strip() + "\n", encoding="utf-8")
print(report)


## 8. Saved Outputs


In [ ]:
outputs = sorted(p.relative_to(OUT_DIR) for p in OUT_DIR.rglob("*") if p.is_file())
print(f"Saved {len(outputs)} files under {OUT_DIR}")
for path in outputs:
    print(path)
